# PenuX — Automated medRxiv Preprint Submission
**Free · Appears in Google Scholar within 24h · No peer review needed**

Run cells 1→5 in order.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install playwright -q
!playwright install chromium
!apt-get install -y -q \
  libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 \
  libnss3 libatk1.0-0 libatk-bridge2.0-0 libdrm2 \
  libxkbcommon0 libasound2 libxshmfence1 libpango-1.0-0 \
  libcairo2 libcups2 libdbus-1-3 libexpat1 libfontconfig1 \
  libglib2.0-0 libnspr4 libx11-6 libx11-xcb1 libxcb1 \
  libxext6 libxrender1 libxtst6
print('✅ Ready')

In [ ]:
# ── Cell 2: Download manuscript from GitHub ────────────────────────────────
import urllib.request, os

URL = 'https://raw.githubusercontent.com/netanelcyber/penuX/claude/pensive-pascal-a0l7a8/cureus_submission/manuscript_medrxiv.docx'
LOCAL = '/tmp/manuscript_penux.docx'
urllib.request.urlretrieve(URL, LOCAL)
print(f'✅ Downloaded: {os.path.getsize(LOCAL):,} bytes')

In [ ]:
# ── Cell 3: Credentials ────────────────────────────────────────────────────
EMAIL    = 'nsh531@gmail.com'
PASSWORD = '318962420'

TITLE = (
    'PenuX: A Comparative Study of 11 Machine Learning and Deep Learning Models '
    'for Early Severity Prediction of Acute Pancreatitis Using Routine Admission '
    'Laboratory Values, with FHIR R4 Integration'
)
ABSTRACT = """Background: Severe Acute Pancreatitis (SAP) carries a mortality rate of 20-30% and requires early risk stratification. Classical scoring systems (Ranson, BISAP, APACHE II) require 24-48 hours of serial laboratory observation and lack EHR integration.

Methods: Retrospective analysis of 722 AP admissions (585 severe / 137 mild; Atlanta 2012) from a single Chinese institution. Eleven models trained on 106 admission laboratory features using 5-fold stratified cross-validation: Logistic Regression, Random Forest, Gradient Boosting, MLP, Residual MLP, Attention MLP, Vanilla LSTM, Stacked LSTM, BiLSTM, LSTM+Attention, CNN-LSTM.

Results: Random Forest achieved AUC=0.877, F1=0.917, sensitivity=96.8%, specificity=38.7% at threshold 0.535. Gradient Boosting: AUC=0.874, sensitivity=97.1%. CNN-LSTM: AUC=0.772, sensitivity=98.6%. Key predictors: calcium, D-dimer, LDH, lactate, hematocrit. A label inversion effect was identified: mild biliary AP cases showed higher WBC/CRP/lipase than severe necrotising AP cases.

Conclusions: Random Forest achieves SAP triage with AUC=0.877 from a single admission blood draw, eliminating the 24-48 hour observation window of classical scoring systems. Open-source platform with FHIR R4, HL7 v2.x, and Israeli HIS (Camelion) integration available at https://penux.uk"""

print(f'✅ Title: {TITLE[:60]}...')
print(f'✅ Abstract: {len(ABSTRACT.split())} words')

In [ ]:
# ── Cell 4: Create account + sign in to medRxiv ───────────────────────────
import asyncio
from playwright.async_api import async_playwright
from IPython.display import Image, display

async def screenshot(page, path):
    await page.screenshot(path=path, full_page=False)
    display(Image(path))

async def signin():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage', '--disable-gpu']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36',
            viewport={'width': 1280, 'height': 900}
        )
        page = await ctx.new_page()

        print('Step 1: Loading medRxiv submission portal...')
        await page.goto('https://submit.medrxiv.org', wait_until='networkidle', timeout=30000)
        await screenshot(page, '/tmp/medrxiv_1_home.png')
        print(f'URL: {page.url}')

        # Try sign in
        print('Step 2: Signing in...')
        try:
            await page.click('text=Log In, text=Sign In, a[href*="login"]', timeout=5000)
            await page.wait_for_load_state('networkidle')
        except:
            # Try direct login URL
            await page.goto('https://submit.medrxiv.org/login', wait_until='networkidle', timeout=20000)
        await screenshot(page, '/tmp/medrxiv_2_login.png')

        # Fill login form
        email_selectors = [
            'input[name="email"]', 'input[type="email"]',
            'input[name="login"]', '#login_email', '#email'
        ]
        for sel in email_selectors:
            try:
                await page.fill(sel, EMAIL, timeout=2000)
                print(f'✅ Email filled via {sel}')
                break
            except: pass

        pass_selectors = [
            'input[name="password"]', 'input[type="password"]', '#password'
        ]
        for sel in pass_selectors:
            try:
                await page.fill(sel, PASSWORD, timeout=2000)
                print(f'✅ Password filled via {sel}')
                break
            except: pass

        await screenshot(page, '/tmp/medrxiv_3_filled.png')

        try:
            await page.click('button[type="submit"], input[type="submit"]', timeout=5000)
            await page.wait_for_load_state('networkidle', timeout=15000)
            await screenshot(page, '/tmp/medrxiv_4_after_login.png')
            print(f'After login URL: {page.url}')
        except Exception as e:
            print(f'Login click: {e}')

        await browser.close()

await signin()

In [ ]:
# ── Cell 5: Submit manuscript ──────────────────────────────────────────────
async def submit():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage', '--disable-gpu']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36',
            viewport={'width': 1280, 'height': 900}
        )
        page = await ctx.new_page()

        # Sign in
        await page.goto('https://submit.medrxiv.org', wait_until='networkidle', timeout=30000)
        try:
            await page.click('text=Log In', timeout=3000)
            await page.wait_for_load_state('networkidle')
        except:
            await page.goto('https://submit.medrxiv.org/login', wait_until='networkidle', timeout=20000)

        for sel in ['input[name="email"]', 'input[type="email"]', '#email']:
            try: await page.fill(sel, EMAIL, timeout=2000); break
            except: pass
        for sel in ['input[name="password"]', 'input[type="password"]', '#password']:
            try: await page.fill(sel, PASSWORD, timeout=2000); break
            except: pass
        try:
            await page.click('button[type="submit"], input[type="submit"]', timeout=5000)
            await page.wait_for_load_state('networkidle', timeout=15000)
            print(f'Signed in. URL: {page.url}')
        except Exception as e:
            print(f'Login: {e}')

        # Start new submission
        print('Starting new submission...')
        try:
            await page.click('text=New Submission, text=Submit, a[href*="submit"]', timeout=5000)
            await page.wait_for_load_state('networkidle')
        except:
            await page.goto('https://submit.medrxiv.org/submit', wait_until='networkidle', timeout=20000)
        await screenshot(page, '/tmp/medrxiv_submit1.png')

        # Fill title
        for sel in ['input[name="title"]', '#title', 'textarea[name="title"]']:
            try:
                await page.fill(sel, TITLE, timeout=3000)
                print('✅ Title filled')
                break
            except: pass

        # Fill abstract
        for sel in ['textarea[name="abstract"]', '#abstract', 'textarea[name="abs"]']:
            try:
                await page.fill(sel, ABSTRACT, timeout=3000)
                print('✅ Abstract filled')
                break
            except: pass

        # Upload manuscript file
        try:
            file_input = page.locator('input[type="file"]').first
            await file_input.set_input_files('/tmp/manuscript_penux.docx')
            print('✅ Manuscript file uploaded')
        except Exception as e:
            print(f'⚠️  File upload: {e}')

        await screenshot(page, '/tmp/medrxiv_submit2_filled.png')
        print('\n⚠️  Review screenshots above. Run Cell 6 to submit.')
        await browser.close()

await submit()

In [ ]:
# ── Cell 6: FINAL SUBMIT ──────────────────────────────────────────────────
CONFIRMED = True

async def final():
    if not CONFIRMED:
        print('Set CONFIRMED = True first')
        return
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage', '--disable-gpu']
        )
        ctx = await browser.new_context(ignore_https_errors=True, viewport={'width': 1280, 'height': 900})
        page = await ctx.new_page()

        # Full sign-in + fill + submit sequence
        await page.goto('https://submit.medrxiv.org', wait_until='networkidle', timeout=30000)
        try:
            await page.click('text=Log In', timeout=3000)
            await page.wait_for_load_state('networkidle')
        except:
            await page.goto('https://submit.medrxiv.org/login', wait_until='networkidle', timeout=20000)

        for sel in ['input[name="email"]', 'input[type="email"]', '#email']:
            try: await page.fill(sel, EMAIL, timeout=2000); break
            except: pass
        for sel in ['input[name="password"]', 'input[type="password"]', '#password']:
            try: await page.fill(sel, PASSWORD, timeout=2000); break
            except: pass
        await page.click('button[type="submit"], input[type="submit"]', timeout=5000)
        await page.wait_for_load_state('networkidle', timeout=15000)

        try:
            await page.click('text=New Submission, text=Submit', timeout=5000)
            await page.wait_for_load_state('networkidle')
        except:
            await page.goto('https://submit.medrxiv.org/submit', wait_until='networkidle', timeout=20000)

        for sel in ['input[name="title"]', '#title']:
            try: await page.fill(sel, TITLE, timeout=3000); break
            except: pass
        for sel in ['textarea[name="abstract"]', '#abstract']:
            try: await page.fill(sel, ABSTRACT, timeout=3000); break
            except: pass
        try:
            await page.locator('input[type="file"]').first.set_input_files('/tmp/manuscript_penux.docx')
        except: pass

        # Final submit click
        try:
            await page.click('button:has-text("Submit"), input[type="submit"]', timeout=8000)
            await page.wait_for_load_state('networkidle', timeout=30000)
            await screenshot(page, '/tmp/medrxiv_final.png')
            print(f'✅ SUBMITTED! URL: {page.url}')
        except Exception as e:
            await screenshot(page, '/tmp/medrxiv_error.png')
            print(f'⚠️  Submit error: {e}')

        await browser.close()

await final()